In [1]:
# =============================================================================
#  STANDALONE: base stitching setup + EXP26 (answer bottleneck) + EXP27 (facts)
#
#  This notebook now contains everything. Nothing else needs to be open.
#
#  ORDER OF OPERATIONS
#    1. Run the dependency cells below (the first four), then RESTART THE KERNEL.
#    2. Paste your Hugging Face token into the login cell (Gemma is gated).
#       The placeholder is hf_PASTE_YOUR_TOKEN_HERE — no real token ships in this file.
#    3. Run everything from the imports cell down.
#
#  CONFIG: the config cell already has SMOKE_TEST = False, i.e. FULL sizes --
#  N_ARITH_TRAIN = 3000, N_ARITH_EVAL = 2000 (~720 unsolvable), TASK_SEEDS = 5,
#  TASK_EPOCHS = 6, BOOT_B = 10000. Flip SMOKE_TEST = True only for a ~5 min
#  plumbing check; flip it back before generating anything for the paper.
#
#  EXP26/EXP27 defaults are sized to match: the writer trains 8 epochs over the
#  1500-item writer split (~750 steps), and the matched-budget task map it is
#  compared against uses the SAME split and epoch count, so that ratio is fair.
#  EXP27 auto-raises its epoch count to clear FACT_MIN_STEPS = 400.
#
#  What is included from the base suite (EXP1 activation patching is NOT — the
#  layer pair SINGLE_PAIR = (20, 34) is hardcoded in the config cell, so the
#  patching sweep is not needed and would only cost GPU time):
#    deps -> imports/config -> HF login -> stats helpers -> shared helpers ->
#    load 9B + 2B -> EXP2 setup (states, recon map, unsolvable bin) ->
#    EXP3 (task maps, 5 seeds) -> EXP2+3 eval (first_token_confer / full_confer)
#  Then: EXP26, EXP27, save.
# =============================================================================
print("read the header above, then run the four dependency cells and restart the kernel")


read the header above, then run the four dependency cells and restart the kernel


In [2]:
# === CELL 1: Dependencies — run cells 0-3 once, in order, then RESTART KERNEL ===
# One resolved install so pip solves versions a SINGLE time, before any model is loaded:
#   * transformers is pinned to the version the model code targets (4.46.3)
#   * sae_lens (for EXP16 / EXP20 / EXP21) is installed HERE, not mid-run, so it can't retug
#     torch/transformers after the models are already sitting in memory.
# The cu128 torch fix in the NEXT cell runs AFTER this line, so on Blackwell / sm_120 pods it
# always wins over whatever torch sae_lens's resolver pulls. (If pip reports a hard transformers
# conflict from sae_lens, drop sae_lens from this line, `pip install sae_lens` on its own, then
# re-run this pinned line so transformers is restored to 4.46.3.)
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer sae_lens


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
notebook 6.5.5 requires pyzmq<25,>=17, but you have pyzmq 26.0.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
!pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 197.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 285.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 191.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 273.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 219.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 588.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 272.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 634.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 319.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 276.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 277.4 MB/s

In [4]:
!pip uninstall -y torchvision torchaudio

Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124


In [6]:
pip install -q -U "typing_extensions>=4.10"


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# verify the resolved stack BEFORE restarting (catch a bad resolve early, not 40 min into a run)
import torch
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))  # want (12, 0) + '+cu128' on Blackwell
import transformers, sae_lens
print("transformers:", transformers.__version__, "(want 4.46.3)  | sae_lens:", sae_lens.__version__)


torch: 2.11.0+cu128 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)  | sae_lens: 6.5.3


In [2]:
# === CELL 2:imports, set_submodule shim, global config (VRAM/compute switches, primarily smoke test) ===
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
MODEL_2B = "google/gemma-2-2b"
MODEL_9B = "google/gemma-2-9b"

# Smoke test lever, (True for 5 min test eval False for full eval)
SMOKE_TEST = False            

# validated layers (Different for each model pair)
SINGLE_PAIR = (20, 34)
LAYER_PAIRS = [(18, 31), (20, 34), (22, 37), (24, 40)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 20, 24, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 300, 3000, 2000      # ~720 unsolvable muladd
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 150, 1319, [1.0]    # full GSM8K test set
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6                # 5 seeds for the task map
    BOOT_B = 10000

ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8
RESULTS = {}   # everything defensible gets concentrated here and printed at the end
print("SMOKE_TEST =", SMOKE_TEST)

SMOKE_TEST = False


In [ ]:
# === CELL 3: Hugging Face login (Gemma is gated) ===
from huggingface_hub import login
login("")

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook
 
def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m
 
def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2
 
# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
 
# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    m = _re.search(r"####\s*(-?[\d,.]+)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    m = _re.search(r"answer is\s*(-?[\d,.]+)", text, _re.IGNORECASE)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    nums = _re.findall(r"-?[\d,.]+", text)
    if nums:
        try: return float(nums[-1].rstrip(".").replace(",", ""))
        except ValueError: return None
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None
 
# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top
 
# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)
 
@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok
 
print("helpers defined")

helpers defined


In [7]:
# === CELL 6: load both models once; donor in bf16 by default (4-bit optional) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
 
# QUANTIZE_9B: set True only when the donor won't fit in bf16 (e.g. the original Gemma-2-9B on
# a small card). For 7-8B donors (Qwen-7B ~15GB, Llama-8B ~16GB) on a 20GB+ card, keep this
# False — bf16 is correct and avoids a serious failure mode: under 4-bit, this transformers/bnb
# build runs unquantized layers in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) ->
# NaN logits -> argmax collapses to token 0 ('!'). bf16 has the exponent range to avoid this.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
 
model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x2B layers,",
      model_9b.config.num_hidden_layers, "x9B layers |",
      "9B quantized" if QUANTIZE_9B else "9B bf16")
 
# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "9B produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl
 
# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

loaded: 26 x2B layers, 42 x9B layers | 9B bf16


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [8]:
# === CELL 8: EXP2 setup — arithmetic states, native correctness, reconstruction map ===
train = gen_arith(tokenizer, N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")

X9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in train]); X9t = X9t[L9_SINGLE]
X9e, nine_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in evalp]); X9e = X9e[L9_SINGLE]
keep = [i for i in range(len(evalp)) if nine_top[i] == evalp[i]["tok"]]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
print(f"  kept {len(evalp)} the 9B solves")

X2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in train]); X2t = X2t[L2_SINGLE]
X2e, two_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in evalp]); X2e = X2e[L2_SINGLE]
solvable = [two_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv = [i for i in range(len(evalp)) if solvable[i]]
print(f"  2B natively solves {len(solv)}/{len(evalp)} (unsolvable bin n={len(unsolv)})")

mu9, mu2, Wr = fit_ridge(X9t, X2t)
recon_map = (mu9.to(DEVICE), mu2.to(DEVICE), Wr.to(DEVICE))
def map_recon(x9): m9, m2, W = recon_map; return (x9.to(DEVICE)-m9)@W + m2

arith: 3000 train, 2000 eval
  kept 1689 the 9B solves
  2B natively solves 1051/1689 (unsolvable bin n=638)


In [9]:
# === CELL 9: EXP3 — train the task-supervised map, 5 seeds (the only stochastic part) ===
mu9d, mu2d = mu9.to(DEVICE), mu2.to(DEVICE)
task_maps = []
model_2b.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu2.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                x9 = X9t[sub].to(DEVICE)
                _graft["vec"] = (x9 - mu9d) @ W + b
                ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}")
model_2b.requires_grad_(True)

  seed 0: final batch CE 0.008
  seed 1: final batch CE 0.150
  seed 2: final batch CE 0.021
  seed 3: final batch CE 0.029
  seed 4: final batch CE 0.015


Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): Gemma2RotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNorm((2304,), eps

In [10]:
# === CELL 10: EXP2+3 eval — reconstruct vs task, first-token AND full-answer, with CIs ===
@torch.inference_mode()
def first_token_confer(map_fn, idxs):
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            _graft["vec"] = map_fn(X9e[sub])
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
def full_confer(map_fn, idxs):
    vecs = list(map_fn(X9e[idxs]).cpu())
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=vecs)
 
def map_task(i):
    W, b = task_maps[i]
    return lambda x9: (x9.to(DEVICE)-mu9d)@W + b
shuf = torch.randperm(len(evalp))
@torch.inference_mode()
def shuffle_confer(idxs):  # task map fed the WRONG problem's 9B state (specificity control)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch); out = []
    W, b = task_maps[0]
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            donor = X9e[shuf[i:i+len(sub)]].to(DEVICE)
            _graft["vec"] = (donor - mu9d)@W + b
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
 
# recon_cos (continuous metric -> bootstrap CI); task map should sit much lower
rc_recon = F.cosine_similarity(map_recon(X9e).cpu(), X2e, dim=1).numpy()
rc_task = F.cosine_similarity(((X9e.to(DEVICE)-mu9d)@task_maps[0][0]+task_maps[0][1]).cpu(), X2e, dim=1).numpy()
arr = {"recon_cos_recon": fmt(bootstrap_ci(rc_recon)),
       "recon_cos_task": fmt(bootstrap_ci(rc_task))}
 
# headline bin: UNSOLVABLE — full-answer + the 5-seed treatment live here
if unsolv:
    arr["native_unsolv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in unsolv], vecs=None)))
    arr["recon_unsolv_full"] = fmt(wilson_bools(full_confer(map_recon, unsolv)))
    arr["recon_unsolv_first"] = fmt(wilson_bools(first_token_confer(map_recon, unsolv)))
    seed_full = [float(np.mean(full_confer(map_task(s), unsolv))) for s in range(len(task_maps))]
    arr["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
    arr["task_unsolv_first_pooled"] = fmt(wilson_bools(first_token_confer(map_task(0), unsolv)))
    arr["shuffle_unsolv_first"] = fmt(wilson_bools(shuffle_confer(unsolv)))
# sanity bin: SOLVABLE — first-token + full-answer (recon and task seed 0)
if solv:
    arr["native_solv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in solv], vecs=None)))
    arr["recon_solv_first"] = fmt(wilson_bools(first_token_confer(map_recon, solv)))
    arr["recon_solv_full"] = fmt(wilson_bools(full_confer(map_recon, solv)))
    arr["task_solv_first"] = fmt(wilson_bools(first_token_confer(map_task(0), solv)))
    arr["task_solv_full"] = fmt(wilson_bools(full_confer(map_task(0), solv)))
RESULTS["arithmetic"] = arr
print("EXP2/3:", json.dumps(arr, indent=2))

EXP2/3: {
  "recon_cos_recon": "0.981 [0.980, 0.982]",
  "recon_cos_task": "0.219 [0.217, 0.222]",
  "native_unsolv_full": "0.003 [0.001, 0.011]",
  "recon_unsolv_full": "0.034 [0.023, 0.052]",
  "recon_unsolv_first": "0.191 [0.163, 0.224]",
  "task_unsolv_full_acrossseed": "0.127 [0.105, 0.148]",
  "task_unsolv_first_pooled": "0.886 [0.859, 0.908]",
  "shuffle_unsolv_first": "0.102 [0.081, 0.128]",
  "native_solv_full": "0.122 [0.103, 0.143]",
  "recon_solv_first": "0.925 [0.907, 0.939]",
  "recon_solv_full": "0.116 [0.098, 0.137]",
  "task_solv_first": "0.951 [0.937, 0.963]",
  "task_solv_full": "0.111 [0.094, 0.132]"
}


In [11]:
# === CELL E26: EXP26 — ANSWER-BOTTLENECK READER/WRITER (sufficiency of answer content) ===
# EXP17 established NECESSITY: erase the answer subspace from the task map's output and
# conferral collapses. This tests SUFFICIENCY from the other side: force the channel to carry
# nothing but the donor's predicted answer scores, and ask how much conferral survives.
#
#   h_D  --[frozen reader g]-->  z in R^K  --[learned writer B]-->  graft in R^{d2}
#
# NOTE ON WHAT THE BOTTLENECK ACTUALLY IS. With BN_SOFTMAX=True (default), z is a point on the
# K-simplex: K-1 CONTINUOUS coordinates, a smooth function of h_D. It is NOT a discrete class
# label, and runner-up mass can carry information beyond the argmax. The one-hot GOLD condition
# is the only strictly discrete channel. Set BN_SOFTMAX=False for raw logits (a wider channel).
#
# Outcomes:
#   * recovers most of the matched task map's conferral => answer scores alone reproduce it;
#     with EXP17 that makes answer content necessary AND approximately sufficient.
#   * poor, but GOLD writer good => the read-out step is the loss, not the write step.
#   * both poor => failure is recipient-side (the writer), NOT evidence against read-out.
#
# REUSES (run the base notebook through the EXP2+3 EVAL cell — the one that defines map_task,
# first_token_confer, full_confer — not merely EXP3): train, evalp, unsolv, X9t, X2t, X9e,
# task_maps, map_task, map_recon, mu9d, first_token_confer, full_confer,
# arith_fullanswer_correct, patch_vec_batch, _graft, left_pad, wilson_bools, fmt, fit_ridge,
# L2_SINGLE, ARITH_BATCH, DEVICE, RESULTS, tokenizer, model_2b.

RUN_BOTTLENECK   = globals().get("RUN_BOTTLENECK", True)
BN_READER_FRAC   = globals().get("BN_READER_FRAC", 0.5)
BN_READER_STEPS  = globals().get("BN_READER_STEPS", 600)
BN_READER_LR     = globals().get("BN_READER_LR", 1e-2)
BN_WRITER_EPOCHS = globals().get("BN_WRITER_EPOCHS", 8)
BN_WRITER_LR     = globals().get("BN_WRITER_LR", 1e-3)
BN_SOFTMAX       = globals().get("BN_SOFTMAX", True)
BN_MATCHED_TASK  = globals().get("BN_MATCHED_TASK", True)   # task map on the SAME writer split
BN_SEED          = globals().get("BN_SEED", 0)

if RUN_BOTTLENECK and unsolv:
    import math
    from fractions import Fraction
    D2 = X2t.shape[1]

    # ---- answer classes from TRAIN ONLY (eval labels must not choose the bottleneck width) ----
    TOKSET  = sorted({p["tok"] for p in train})
    tok2cls = {t: i for i, t in enumerate(TOKSET)}
    K = len(TOKSET)
    _oov = sum(1 for j in unsolv if evalp[j]["tok"] not in tok2cls)
    print(f"E26: K = {K} answer classes from train "
          f"({[tokenizer.decode([t]) for t in TOKSET[:12]]}{'...' if K > 12 else ''})")
    print(f"     eval items whose answer token is OUT of the train class set: {_oov}/{len(unsolv)} "
          f"(these can never be conferred by the bottleneck, by construction)")

    _ridx = list(range(len(train))); random.Random(BN_SEED).shuffle(_ridx)
    _cut  = int(len(_ridx) * BN_READER_FRAC)
    R_IDX, W_IDX = _ridx[:_cut], _ridx[_cut:]
    print(f"     reader-train n={len(R_IDX)} | writer-train n={len(W_IDX)} (disjoint) | bin n={len(unsolv)}")

    # ---------- 1. frozen answer READER (reader split only) ----------
    def _train_reader(X, y, steps=BN_READER_STEPS, lr=BN_READER_LR):
        mu = X.mean(0)
        W  = torch.zeros(X.shape[1], K, requires_grad=True)
        b  = torch.zeros(K, requires_grad=True)
        opt = torch.optim.Adam([W, b], lr=lr); A = X - mu
        for _ in range(steps):
            opt.zero_grad(); F.cross_entropy(A @ W + b, y).backward(); opt.step()
        return mu.detach(), W.detach(), b.detach()

    READER = _train_reader(X9t[R_IDX],
                           torch.tensor([tok2cls[train[i]["tok"]] for i in R_IDX]))

    @torch.no_grad()
    def _reader_logits(x9):                       # frozen: no grad path into the reader
        mu, W, b = READER
        return (x9.to(DEVICE) - mu.to(DEVICE)) @ W.to(DEVICE) + b.to(DEVICE)

    def _scores(x9):
        z = _reader_logits(x9)
        return F.softmax(z, -1) if BN_SOFTMAX else z

    _r_pred = _reader_logits(X9e[unsolv]).argmax(-1).cpu().tolist()
    READER_OK = [_r_pred[k] == tok2cls.get(evalp[j]["tok"], -1) for k, j in enumerate(unsolv)]
    READER_ACC = sum(READER_OK) / len(READER_OK)
    print(f"     reader accuracy on the bin = {READER_ACC:.4f}  (hard ceiling for the system)")

    # ---------- 2. WRITER trained through the frozen recipient ----------
    def _train_writer(z_fn, tag):
        torch.manual_seed(BN_SEED)
        Bw = torch.empty(K, D2, device=DEVICE)
        with torch.no_grad(): Bw.normal_(0.0, 0.02)
        Bw.requires_grad_(True)
        bw = X2t.mean(0).to(DEVICE).clone().float().requires_grad_(True)
        opt = torch.optim.Adam([Bw, bw], lr=BN_WRITER_LR)
        model_2b.requires_grad_(False)
        handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            for ep in range(BN_WRITER_EPOCHS):
                idx = list(W_IDX); random.Random(1000 + ep).shuffle(idx)
                tot, seen = 0.0, 0
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    graft = z_fn(sub) @ Bw + bw
                    _graft["vec"] = graft
                    ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(logits, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    tot += float(loss.item()) * len(sub); seen += len(sub)
                print(f"     writer[{tag}] epoch {ep}  CE = {tot/max(1,seen):.4f}")
        finally:
            handle.remove(); _graft["vec"] = None
            model_2b.requires_grad_(True)       # restore even if training raised
        return Bw.detach(), bw.detach()

    def _z_pred(sub): return _scores(X9t[sub])
    def _z_gold(sub):
        y = torch.tensor([tok2cls[train[k]["tok"]] for k in sub], device=DEVICE)
        return F.one_hot(y, K).float()
    def _z_unif(sub):                                   # bias-only / no-information control
        return torch.full((len(sub), K), 1.0 / K, device=DEVICE)

    BW_PRED = _train_writer(_z_pred, "pred")
    BW_GOLD = _train_writer(_z_gold, "gold")
    BW_UNIF = _train_writer(_z_unif, "uniform")

    # ---------- 3. eval helpers (index-aware; batch-aligned by construction) ----------
    @torch.inference_mode()
    def _confer_vecfn(vec_fn, idxs):
        """vec_fn(sub) -> [len(sub), d2] on DEVICE. Grafts per batch, like the base helpers."""
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i + ARITH_BATCH]
                v = vec_fn(sub)
                assert v.shape[0] == len(sub), f"graft batch {v.shape[0]} != {len(sub)}"
                _graft["vec"] = v
                ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return out

    @torch.inference_mode()
    def _emits_which(vec_fn, idxs, partner):
        """Top token vs the PARTNER item's answer (did the channel deliver the other answer?)."""
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i + ARITH_BATCH]
                _graft["vec"] = vec_fn(sub)
                ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == evalp[partner[sub[k]]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return out

    def _full_vecfn(vec_fn, idxs):
        V = torch.cat([vec_fn(idxs[i:i + ARITH_BATCH]).detach().cpu()
                       for i in range(0, len(idxs), ARITH_BATCH)], 0)
        return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=list(V))

    def _mk(mode, Bb):
        Bw, bw = Bb
        def f(sub):
            if mode == "pred":  z = _scores(X9e[sub])
            elif mode == "unif": z = torch.full((len(sub), K), 1.0 / K, device=DEVICE)
            elif mode == "gold":
                y = torch.tensor([tok2cls.get(evalp[j]["tok"], 0) for j in sub], device=DEVICE)
                z = F.one_hot(y, K).float()
            elif mode == "shuf": z = _scores(X9e[[PARTNER[j] for j in sub]])
            else: raise ValueError(mode)
            return z @ Bw + bw
        return f

    # ---- derangement with DISTINCT answer tokens (no fixed points, no class collisions) ----
    def _partner_map(idxs, seed):
        rng = random.Random(seed)
        pool = list(idxs)
        for _ in range(200):
            perm = pool[:]; rng.shuffle(perm)
            bad = [k for k in range(len(pool))
                   if evalp[perm[k]]["tok"] == evalp[pool[k]]["tok"]]
            if not bad: return dict(zip(pool, perm)), 0
            for k in bad:                                    # greedy repair
                for m2 in range(len(pool)):
                    if (evalp[perm[m2]]["tok"] != evalp[pool[k]]["tok"] and
                        evalp[perm[k]]["tok"]  != evalp[pool[m2]]["tok"]):
                        perm[k], perm[m2] = perm[m2], perm[k]; break
            bad = [k for k in range(len(pool))
                   if evalp[perm[k]]["tok"] == evalp[pool[k]]["tok"]]
            if not bad: return dict(zip(pool, perm)), 0
        return dict(zip(pool, perm)), len(bad)
    PARTNER, _n_collide = _partner_map(unsolv, BN_SEED + 7)
    print(f"     shuffled pairing: {len(PARTNER)} pairs, {_n_collide} unavoidable same-token collisions")

    # ---------- 4. matched-budget task map (same writer split, same epochs) ----------
    MATCHED = None
    if BN_MATCHED_TASK:
        torch.manual_seed(BN_SEED)
        _mu9, _mu2, _Wr0 = fit_ridge(X9t[W_IDX], X2t[W_IDX])
        Wm = _Wr0.clone().to(DEVICE).requires_grad_(True)
        bm = _mu2.clone().to(DEVICE).float().requires_grad_(True)
        _mu9d = _mu9.to(DEVICE)
        opt = torch.optim.Adam([Wm, bm], lr=BN_WRITER_LR)
        model_2b.requires_grad_(False)
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            for ep in range(BN_WRITER_EPOCHS):
                idx = list(W_IDX); random.Random(2000 + ep).shuffle(idx)
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    _graft["vec"] = (X9t[sub].to(DEVICE) - _mu9d) @ Wm + bm
                    ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(logits, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
        finally:
            h.remove(); _graft["vec"] = None
            model_2b.requires_grad_(True)
        Wm, bm = Wm.detach(), bm.detach()
        MATCHED = lambda sub: (X9e[sub].to(DEVICE) - _mu9d) @ Wm + bm
        print("     matched-budget task map trained (same split + epochs as the writer)")

    # ---------- 5. conditions ----------
    BN_pred_first = _confer_vecfn(_mk("pred", BW_PRED), unsolv)
    BN_gold_first = _confer_vecfn(_mk("gold", BW_GOLD), unsolv)
    # items whose answer token has no train class get a one-hot at the WRONG class in gold mode,
    # so they can never be conferred. Report the ceiling on the in-vocab subset as well.
    _IV = [k for k, j in enumerate(unsolv) if evalp[j]["tok"] in tok2cls]
    BN_gold_first_iv = [BN_gold_first[k] for k in _IV]
    BN_unif_first = _confer_vecfn(_mk("unif", BW_UNIF), unsolv)
    BN_shuf_first = _confer_vecfn(_mk("shuf", BW_PRED), unsolv)
    BN_shuf_other = _emits_which(_mk("shuf", BW_PRED), unsolv, PARTNER)
    BN_pred_full  = _full_vecfn(_mk("pred", BW_PRED), unsolv)
    BN_gold_full  = _full_vecfn(_mk("gold", BW_GOLD), unsolv)

    TASK_first  = first_token_confer(map_task(0), unsolv)
    TASK_full   = full_confer(map_task(0), unsolv)
    RECON_first = first_token_confer(map_recon, unsolv)
    MATCH_first = _confer_vecfn(MATCHED, unsolv) if MATCHED else None

    def _mcnemar(a, b):
        """Exact two-sided McNemar. Fraction keeps big binomials off the float path."""
        n01 = sum(1 for x, y in zip(a, b) if (not x) and y)
        n10 = sum(1 for x, y in zip(a, b) if x and (not y))
        n = n01 + n10
        if n == 0: return n01, n10, 1.0
        k = min(n01, n10)
        p = Fraction(2 * sum(math.comb(n, i) for i in range(k + 1)), 2 ** n)
        return n01, n10, min(1.0, float(p))

    _rate = lambda v: sum(v) / len(v)
    _c_bn, _c_task, _c_gold = _rate(BN_pred_first), _rate(TASK_first), _rate(BN_gold_first)
    _c_match = _rate(MATCH_first) if MATCH_first else None
    _n01, _n10, _p = _mcnemar(TASK_first, BN_pred_first)

    RESULTS["answer_bottleneck"] = {
        "_what": ("h_D -> FROZEN linear answer reader -> K scores -> LEARNED writer -> graft. "
                  "The writer sees only the K scores. Tests whether answer content alone "
                  "SUFFICES to reproduce conferral."),
        "_bottleneck_width_note": ("softmax scores are K-1 continuous coordinates, not a discrete "
                                   "label; only the gold one-hot condition is strictly discrete"),
        "K_classes": K, "scores_are_softmax": bool(BN_SOFTMAX),
        "n_reader_train": len(R_IDX), "n_writer_train": len(W_IDX), "n_eval_bin": len(unsolv),
        "n_eval_answers_outside_train_classes": _oov,
        "reader_acc_on_bin": fmt(wilson_bools(READER_OK)),

        "bottleneck_pred_first":      fmt(wilson_bools(BN_pred_first)),
        "bottleneck_gold_first":      fmt(wilson_bools(BN_gold_first)),
        "bottleneck_gold_first_invocab": (fmt(wilson_bools(BN_gold_first_iv))
                                          if len(BN_gold_first_iv) != len(BN_gold_first)
                                          else "same as above (no out-of-vocab answers)"),
        "bottleneck_uniform_first":   fmt(wilson_bools(BN_unif_first)),
        "bottleneck_shuffled_first":  fmt(wilson_bools(BN_shuf_first)),
        "bottleneck_shuffled_emits_PARTNER_answer": fmt(wilson_bools(BN_shuf_other)),
        "bottleneck_pred_full":       fmt(wilson_bools(BN_pred_full)),
        "bottleneck_gold_full":       fmt(wilson_bools(BN_gold_full)),

        "_ref_task_first_FULL_data":   fmt(wilson_bools(TASK_first)),
        "_ref_task_first_MATCHED_data": fmt(wilson_bools(MATCH_first)) if MATCH_first else "n/a",
        "_ref_task_full":              fmt(wilson_bools(TASK_full)),
        "_ref_recon_first":            fmt(wilson_bools(RECON_first)),
        "_ref_native_first":           "0.000 [by construction]",

        "recovery_vs_matched_task": round(_c_bn / _c_match, 4) if _c_match else None,
        "recovery_vs_full_task":    round(_c_bn / _c_task, 4) if _c_task else None,
        "_note_recovery": ("compare against the MATCHED map: the full-data task map saw twice the "
                           "training items the writer did, so the full-data ratio is confounded"),
        "_note_ceiling": ("conferral is bounded by the reader: items the reader gets wrong can "
                          "only be conferred by chance. Read bottleneck_pred_first against "
                          "reader_acc_on_bin and against bottleneck_uniform_first (bias-only)."),
        "paired_task_vs_bottleneck": {"task_only": _n10, "bottleneck_only": _n01, "mcnemar_p": _p},
        "shuffled_pairing": {"n_pairs": len(PARTNER), "unavoidable_same_token": _n_collide,
                             "_note": "derangement with distinct answer tokens; no fixed points"},
    }
    print("EXP26 answer bottleneck:", json.dumps(RESULTS["answer_bottleneck"], indent=2))
else:
    print("EXP26 skipped (RUN_BOTTLENECK=False or empty unsolvable bin).")


E26: K = 9 answer classes from train (['1', '2', '3', '5', '4', '9', '6', '8', '7'])
     eval items whose answer token is OUT of the train class set: 0/638 (these can never be conferred by the bottleneck, by construction)
     reader-train n=1500 | writer-train n=1500 (disjoint) | bin n=638
     reader accuracy on the bin = 0.8339  (hard ceiling for the system)
     writer[pred] epoch 0  CE = 1.6999
     writer[pred] epoch 1  CE = 1.3204
     writer[pred] epoch 2  CE = 1.0710
     writer[pred] epoch 3  CE = 0.8834
     writer[pred] epoch 4  CE = 0.7501
     writer[pred] epoch 5  CE = 0.6575
     writer[pred] epoch 6  CE = 0.5965
     writer[pred] epoch 7  CE = 0.5587
     writer[gold] epoch 0  CE = 1.6747
     writer[gold] epoch 1  CE = 1.2425
     writer[gold] epoch 2  CE = 0.9278
     writer[gold] epoch 3  CE = 0.6701
     writer[gold] epoch 4  CE = 0.4636
     writer[gold] epoch 5  CE = 0.3057
     writer[gold] epoch 6  CE = 0.1954
     writer[gold] epoch 7  CE = 0.1237
     writer

In [14]:
from datasets import load_dataset
import json
FEWSHOT = ("Question: In which country is the city of Osaka?\nAnswer: Japan\n\n"
           "Question: Who wrote the play Hamlet?\nAnswer: Shakespeare\n\n"
           "Question: What is the chemical symbol for gold?\nAnswer: Au\n\n")
ds = load_dataset("trivia_qa", "rc.nocontext", split="validation")
print(f"loaded {len(ds)} trivia questions")
rows, seen = [], set()
for r in ds:
    q = (r["question"] or "").strip()
    a = (r["answer"]["value"] or "").strip()
    if not q or not a or len(a.split()) > 3:
        continue
    p = FEWSHOT + f"Question: {q}\nAnswer:"
    if p in seen:
        continue
    seen.add(p); rows.append({"prompt": p, "answer": a})
    if len(rows) >= 8000:
        break
with open("facts.jsonl", "w") as fh:
    for r in rows:
        fh.write(json.dumps(r) + "\n")
print(f"wrote {len(rows)} facts -> facts.jsonl")
FACT_SOURCE = "facts.jsonl"

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc.nocontext/train-00000-of-00001.parque(…):   0%|          | 0.00/55.4M [00:00<?, ?B/s]

rc.nocontext/validation-00000-of-00001.p(…):   0%|          | 0.00/7.34M [00:00<?, ?B/s]

rc.nocontext/test-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

loaded 17944 trivia questions
wrote 8000 facts -> facts.jsonl


In [15]:
# === CELL E27: EXP27 — NATURALISTIC TASK (factual recall), full channel battery ===
# The paper's Limitations concede that three of four task families are synthetic and that
# arithmetic carries most of the weight. Factual recall is naturalistic, has a short
# well-defined answer, and is solved by the donor INSIDE a single forward pass -- so it meets
# the presence law's preconditions while testing a different KIND of content: retrieved
# knowledge rather than computed answers.
#
# Same graft site and same bin rule as arithmetic: {donor gets the first answer token right}
# AND {recipient gets it wrong}. Native first-token accuracy on that bin is 0 by construction.
#
# >>> DATA. FACT_SOURCE="builtin" is a SMOKE TEST ONLY. A 2B model knows most capitals, so the
# >>> bin will be tens of items, far too small for the paper. For real numbers set FACT_SOURCE
# >>> to a long-tail JSONL ({"prompt":..., "answer":...} per line) -- CounterFact, LAMA/T-REx,
# >>> or a TriviaQA short-answer subset. The cell prints the bin size and the optimizer step
# >>> count so you can see immediately whether either is too small to trust.
#
# REUSES: states_and_top, left_pad, fit_ridge, patch_vec_batch, _graft, wilson_bools,
# across_seed_ci, fmt, L2_SINGLE, L9_SINGLE, ARITH_BATCH, DEVICE, RESULTS, tokenizer,
# model_2b, model_9b.

RUN_FACTS       = globals().get("RUN_FACTS", True)
FACT_SOURCE     = globals().get("FACT_SOURCE", "builtin")
FACT_TRAIN_FRAC = globals().get("FACT_TRAIN_FRAC", 0.6)
FACT_SEEDS      = globals().get("FACT_SEEDS", [0, 1, 2, 3, 4])
FACT_EPOCHS     = globals().get("FACT_EPOCHS", 6)
FACT_MIN_STEPS  = globals().get("FACT_MIN_STEPS", 400)   # scale epochs up on small data
FACT_LR         = globals().get("FACT_LR", 1e-3)
MAX_NEW_FACT    = globals().get("MAX_NEW_FACT", 12)      # long capitals need room

# Few-shot exemplars are deliberately NOT drawn from the item list below (no answer leakage).
FACT_FEWSHOT = ("The capital of Brazil is Brasilia.\n"
                "The capital of Kenya is Nairobi.\n"
                "The capital of Sweden is Stockholm.\n")

_BUILTIN_CAPITALS = [
    ("Afghanistan","Kabul"),("Albania","Tirana"),("Algeria","Algiers"),("Argentina","Buenos Aires"),
    ("Armenia","Yerevan"),("Australia","Canberra"),("Austria","Vienna"),("Azerbaijan","Baku"),
    ("Bahrain","Manama"),("Bangladesh","Dhaka"),("Belarus","Minsk"),("Belgium","Brussels"),
    ("Bhutan","Thimphu"),("Botswana","Gaborone"),("Bulgaria","Sofia"),
    ("Burkina Faso","Ouagadougou"),("Cambodia","Phnom Penh"),("Canada","Ottawa"),
    ("Chile","Santiago"),("Colombia","Bogota"),("Costa Rica","San Jose"),("Croatia","Zagreb"),
    ("Cuba","Havana"),("Cyprus","Nicosia"),("Denmark","Copenhagen"),("Ecuador","Quito"),
    ("Egypt","Cairo"),("El Salvador","San Salvador"),("Eritrea","Asmara"),("Estonia","Tallinn"),
    ("Ethiopia","Addis Ababa"),("Fiji","Suva"),("Finland","Helsinki"),("France","Paris"),
    ("Gabon","Libreville"),("Georgia","Tbilisi"),("Germany","Berlin"),("Ghana","Accra"),
    ("Greece","Athens"),("Guatemala","Guatemala City"),("Guinea","Conakry"),("Guyana","Georgetown"),
    ("Honduras","Tegucigalpa"),("Hungary","Budapest"),("Iceland","Reykjavik"),("India","New Delhi"),
    ("Indonesia","Jakarta"),("Iran","Tehran"),("Iraq","Baghdad"),("Ireland","Dublin"),
    ("Italy","Rome"),("Jamaica","Kingston"),("Japan","Tokyo"),("Jordan","Amman"),
    ("Kazakhstan","Astana"),("Kuwait","Kuwait City"),("Kyrgyzstan","Bishkek"),
    ("Laos","Vientiane"),("Latvia","Riga"),("Lebanon","Beirut"),("Liberia","Monrovia"),
    ("Libya","Tripoli"),("Lithuania","Vilnius"),("Luxembourg","Luxembourg"),
    ("Madagascar","Antananarivo"),("Malawi","Lilongwe"),("Malaysia","Kuala Lumpur"),
    ("Mali","Bamako"),("Malta","Valletta"),("Mexico","Mexico City"),("Moldova","Chisinau"),
    ("Mongolia","Ulaanbaatar"),("Montenegro","Podgorica"),("Morocco","Rabat"),
    ("Mozambique","Maputo"),("Namibia","Windhoek"),("Nepal","Kathmandu"),
    ("Netherlands","Amsterdam"),("New Zealand","Wellington"),("Nicaragua","Managua"),
    ("Niger","Niamey"),("Nigeria","Abuja"),("North Macedonia","Skopje"),("Norway","Oslo"),
    ("Oman","Muscat"),("Pakistan","Islamabad"),("Panama","Panama City"),
    ("Papua New Guinea","Port Moresby"),("Paraguay","Asuncion"),("Peru","Lima"),
    ("Philippines","Manila"),("Poland","Warsaw"),("Portugal","Lisbon"),("Qatar","Doha"),
    ("Romania","Bucharest"),("Russia","Moscow"),("Rwanda","Kigali"),("Samoa","Apia"),
    ("Saudi Arabia","Riyadh"),("Senegal","Dakar"),("Serbia","Belgrade"),("Sierra Leone","Freetown"),
    ("Singapore","Singapore"),("Slovakia","Bratislava"),("Slovenia","Ljubljana"),
    ("Somalia","Mogadishu"),("Spain","Madrid"),("Sudan","Khartoum"),("Suriname","Paramaribo"),
    ("Switzerland","Bern"),("Syria","Damascus"),("Taiwan","Taipei"),("Tajikistan","Dushanbe"),
    ("Tanzania","Dodoma"),("Thailand","Bangkok"),("Togo","Lome"),("Tunisia","Tunis"),
    ("Turkey","Ankara"),("Turkmenistan","Ashgabat"),("Uganda","Kampala"),("Ukraine","Kyiv"),
    ("Uruguay","Montevideo"),("Uzbekistan","Tashkent"),("Vanuatu","Port Vila"),
    ("Venezuela","Caracas"),("Vietnam","Hanoi"),("Yemen","Sanaa"),("Zambia","Lusaka"),
    ("Zimbabwe","Harare"),
]

FACT_OK = False
if RUN_FACTS:
    import json as _json

    def _first_answer_token(tok, prompt, answer):
        """Prefix-consistent only, like the base _aencode: REJECT items whose prompt+answer
        does not extend the prompt's own tokenization. Splicing them by hand would inject
        sequences the model never produces naturally."""
        p = tok(prompt).input_ids
        f = tok(prompt + " " + answer).input_ids
        if f[:len(p)] != p or len(f) <= len(p):
            return None, None
        j = len(p)
        while j < len(f) and tok.decode([f[j]]).strip() == "":
            j += 1
        if j >= len(f):
            return None, None
        return torch.tensor(f[:j]), f[j]

    if FACT_SOURCE == "builtin":
        _raw = [{"prompt": FACT_FEWSHOT + f"The capital of {s} is", "answer": a}
                for s, a in _BUILTIN_CAPITALS]
        print(f"E27: FACT_SOURCE=builtin -> {len(_raw)} candidates (SMOKE TEST SIZE)")
    else:
        _raw = []
        with open(FACT_SOURCE) as fh:
            for line in fh:
                line = line.strip()
                if line: _raw.append(_json.loads(line))
        print(f"E27: FACT_SOURCE={FACT_SOURCE} -> {len(_raw)} candidates")

    FACTS = []
    for r in _raw:
        ids, tid = _first_answer_token(tokenizer, r["prompt"], r["answer"])
        if tid is not None:
            FACTS.append(dict(prompt=r["prompt"], answer=r["answer"], ids=ids, tok=tid))
    print(f"     {len(_raw)} candidates -> {len(FACTS)} encoded (prefix-consistent)")

    if len(FACTS) < 20:
        print(f"EXP27 STOPPED: only {len(FACTS)} usable items. Supply a larger FACT_SOURCE.")
    else:
        _fi = list(range(len(FACTS))); random.Random(0).shuffle(_fi)
        _cut = max(1, int(len(_fi) * FACT_TRAIN_FRAC))
        FT_TRAIN = [FACTS[i] for i in _fi[:_cut]]
        FT_EVAL  = [FACTS[i] for i in _fi[_cut:]]
        print(f"     train={len(FT_TRAIN)} eval={len(FT_EVAL)}")
        if len(FT_EVAL) >= 5:
            FACT_OK = True
        else:
            print("EXP27 STOPPED: eval split empty. Lower FACT_TRAIN_FRAC or add data.")

if FACT_OK:
    FX9t, _      = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in FT_TRAIN]); FX9t = FX9t[L9_SINGLE]
    FX2t, _      = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_TRAIN]); FX2t = FX2t[L2_SINGLE]
    FX9e, F9_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in FT_EVAL]);  FX9e = FX9e[L9_SINGLE]
    FX2e, F2_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in FT_EVAL]);  FX2e = FX2e[L2_SINGLE]

    F_donor_ok = [F9_top[i] == FT_EVAL[i]["tok"] for i in range(len(FT_EVAL))]
    F_recip_ok = [F2_top[i] == FT_EVAL[i]["tok"] for i in range(len(FT_EVAL))]
    F_UNSOLV = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and not F_recip_ok[i]]
    F_SOLV   = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and F_recip_ok[i]]
    print(f"     donor solves {sum(F_donor_ok)}/{len(FT_EVAL)} | recipient solves {sum(F_recip_ok)}/{len(FT_EVAL)}")
    print(f"     >>> UNSOLVABLE BIN n = {len(F_UNSOLV)}  (need ~150+ for a usable Wilson interval)")
    if len(F_UNSOLV) < 5:
        FACT_OK = False
        print("EXP27 STOPPED: bin too small. Use a longer-tail FACT_SOURCE.")

if FACT_OK:
    _spe = max(1, (len(FT_TRAIN) + ARITH_BATCH - 1) // ARITH_BATCH)
    F_EPOCHS = max(FACT_EPOCHS, -(-FACT_MIN_STEPS // _spe))
    print(f"     {_spe} steps/epoch x {F_EPOCHS} epochs = {_spe*F_EPOCHS} optimizer steps "
          f"(raised from {FACT_EPOCHS} epochs to clear FACT_MIN_STEPS={FACT_MIN_STEPS})")

    Fmu9, Fmu2, FWr = fit_ridge(FX9t, FX2t)
    Fmu9d, Fmu2d, FWr_d = Fmu9.to(DEVICE), Fmu2.to(DEVICE), FWr.to(DEVICE)

    F_task_maps = []
    for sd in FACT_SEEDS:
        torch.manual_seed(sd)
        W = FWr.clone().to(DEVICE).requires_grad_(True)
        b = Fmu2.clone().to(DEVICE).float().requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=FACT_LR)
        model_2b.requires_grad_(False)
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            idx = list(range(len(FT_TRAIN)))
            for ep in range(F_EPOCHS):
                random.Random(100 * sd + ep).shuffle(idx)
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    _graft["vec"] = (FX9t[sub].to(DEVICE) - Fmu9d) @ W + b
                    ids, m = left_pad([FT_TRAIN[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([FT_TRAIN[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(logits, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
        finally:
            h.remove(); _graft["vec"] = None
            model_2b.requires_grad_(True)
        F_task_maps.append((W.detach(), b.detach()))
        print(f"     fact task map seed {sd} trained")

    # ---- index-aware eval helpers: vec_fn(sub) -> [len(sub), d2]. Never returns a
    # ---- full-length tensor, so patch_vec_batch's batch guard can never silently no-op.
    @torch.inference_mode()
    def F_first_confer(vec_fn, idxs):
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        out = []
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i + ARITH_BATCH]
                v = vec_fn(sub)
                assert v.shape[0] == len(sub), f"graft batch {v.shape[0]} != {len(sub)}"
                _graft["vec"] = v
                ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == FT_EVAL[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return out

    def _norm(t):
        return "".join(ch for ch in t.lower() if ch.isascii() and (ch.isalnum() or ch == " ")).strip()

    def F_full_confer(vec_fn, idxs):
        ok, h = [], model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i + ARITH_BATCH]
                _graft["vec"] = vec_fn(sub)
                ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
                gen = model_2b.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                        max_new_tokens=MAX_NEW_FACT, do_sample=False,
                                        pad_token_id=tokenizer.eos_token_id)
                txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
                for k, t in enumerate(txt):
                    gold = _norm(FT_EVAL[sub[k]]["answer"])
                    ok.append(bool(gold) and _norm(t.split("\n")[0]).startswith(gold))
        finally:
            h.remove(); _graft["vec"] = None
        return ok

    @torch.inference_mode()
    def F_native_full(idxs):
        ok = []
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i + ARITH_BATCH]
            ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
            gen = model_2b.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                    max_new_tokens=MAX_NEW_FACT, do_sample=False,
                                    pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for k, t in enumerate(txt):
                gold = _norm(FT_EVAL[sub[k]]["answer"])
                ok.append(bool(gold) and _norm(t.split("\n")[0]).startswith(gold))
        return ok

    F_recon  = lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ FWr_d + Fmu2d
    def F_task(i):
        W, b = F_task_maps[i]
        return lambda sub: (FX9e[sub].to(DEVICE) - Fmu9d) @ W + b
    F_self   = lambda sub: FX2e[sub].to(DEVICE).float()

    # derangement with distinct answer tokens (no fixed points, no trivial collisions)
    def _fpartner(idxs, seed):
        rng = random.Random(seed); pool = list(idxs)
        perm = pool[:]
        for _ in range(200):
            rng.shuffle(perm)
            if all(FT_EVAL[perm[k]]["tok"] != FT_EVAL[pool[k]]["tok"] for k in range(len(pool))):
                return dict(zip(pool, perm)), 0
        bad = sum(1 for k in range(len(pool)) if FT_EVAL[perm[k]]["tok"] == FT_EVAL[pool[k]]["tok"])
        return dict(zip(pool, perm)), bad
    F_PART, _fcol = _fpartner(F_UNSOLV, 7)
    def F_shuf(sub):                       # each item gets its PARTNER's donor state
        part = [F_PART[j] for j in sub]
        W0, b0 = F_task_maps[0]
        return (FX9e[part].to(DEVICE) - Fmu9d) @ W0 + b0

    F_self_first  = F_first_confer(F_self,     F_UNSOLV)
    F_recon_first = F_first_confer(F_recon,    F_UNSOLV)
    F_task_first  = F_first_confer(F_task(0),  F_UNSOLV)
    F_shuf_first  = F_first_confer(F_shuf,     F_UNSOLV)
    F_recon_full  = F_full_confer(F_recon,     F_UNSOLV)
    F_task_full   = F_full_confer(F_task(0),   F_UNSOLV)
    F_nat_full    = F_native_full(F_UNSOLV)
    _seed_first = [sum(F_first_confer(F_task(si), F_UNSOLV)) / len(F_UNSOLV)
                   for si in range(len(F_task_maps))]

    RESULTS["factual_recall"] = {
        "_what": ("naturalistic factual recall, same graft site and same bin rule as arithmetic: "
                  "donor gets the first answer token right, recipient does not."),
        "source": FACT_SOURCE,
        "n_raw_candidates": len(_raw), "n_encoded": len(FACTS),
        "n_train": len(FT_TRAIN), "n_eval": len(FT_EVAL),
        "optimizer_steps_per_map": _spe * F_EPOCHS,
        "donor_first_acc": round(sum(F_donor_ok) / len(FT_EVAL), 4),
        "recipient_first_acc": round(sum(F_recip_ok) / len(FT_EVAL), 4),
        "n_unsolv": len(F_UNSOLV), "n_solv": len(F_SOLV),
        "native_first":    "0.000 [by construction]",
        "native_full":     fmt(wilson_bools(F_nat_full)),
        "selfgraft_first": fmt(wilson_bools(F_self_first)),
        "shuffle_first":   fmt(wilson_bools(F_shuf_first)),
        "recon_first":     fmt(wilson_bools(F_recon_first)),
        "task_first_seed0": fmt(wilson_bools(F_task_first)),
        "task_first_per_seed": [round(v, 4) for v in _seed_first],
        "task_first_acrossseed": fmt(across_seed_ci(_seed_first)) if len(_seed_first) > 1 else "n/a",
        "recon_full": fmt(wilson_bools(F_recon_full)),
        "task_full_seed0": fmt(wilson_bools(F_task_full)),
        "shuffled_pairing": {"n_pairs": len(F_PART), "unavoidable_same_token": _fcol},
        "_warn": ("BUILTIN source is a smoke test; n_unsolv is far too small for the paper. "
                  "Point FACT_SOURCE at a long-tail dataset."
                  if FACT_SOURCE == "builtin" else ""),
    }
    print("EXP27 factual recall:", json.dumps(RESULTS["factual_recall"], indent=2))
elif not RUN_FACTS:
    print("EXP27 skipped (RUN_FACTS=False).")


E27: FACT_SOURCE=facts.jsonl -> 8000 candidates
     8000 candidates -> 8000 encoded (prefix-consistent)
     train=4800 eval=3200
     donor solves 1671/3200 | recipient solves 1292/3200
     >>> UNSOLVABLE BIN n = 471  (need ~150+ for a usable Wilson interval)
     300 steps/epoch x 6 epochs = 1800 optimizer steps (raised from 6 epochs to clear FACT_MIN_STEPS=400)
     fact task map seed 0 trained
     fact task map seed 1 trained
     fact task map seed 2 trained
     fact task map seed 3 trained
     fact task map seed 4 trained
EXP27 factual recall: {
  "_what": "naturalistic factual recall, same graft site and same bin rule as arithmetic: donor gets the first answer token right, recipient does not.",
  "source": "facts.jsonl",
  "n_raw_candidates": 8000,
  "n_encoded": 8000,
  "n_train": 4800,
  "n_eval": 3200,
  "optimizer_steps_per_map": 1800,
  "donor_first_acc": 0.5222,
  "recipient_first_acc": 0.4037,
  "n_unsolv": 471,
  "n_solv": 1200,
  "native_first": "0.000 [by construc

In [16]:
# === CELL S: save results ===
import json, datetime
_out = "stitch_exp26_27_results.json"
_keys = [k for k in ("answer_bottleneck", "factual_recall") if k in RESULTS]
with open(_out, "w") as fh:
    json.dump({k: RESULTS[k] for k in _keys}, fh, indent=2)
print("wrote", _out, "with", _keys, "at", datetime.datetime.now().isoformat(timespec="seconds"))


wrote stitch_exp26_27_results.json with ['answer_bottleneck', 'factual_recall'] at 2026-09-09T04:46:11
